# Domain G: GLM-Based Encoding Analysis

This notebook addresses three questions using pre-fitted ridge-regularized GLM encoding models stored in the zarr multimodal data:
- **G1:** Do cell types differ in the relative balance of visual vs. state-driven activity?
- **G2:** What do condition-specific GLM kernels reveal about temporal encoding across cell types?
- **G3:** Can GLM residuals (y − y_hat) reveal unmodeled computation?

**Data:** Zarr multimodal stores containing GLM coefficients, scores, y (observed), y_hat (predicted), y_hat_visual, and y_hat_state per cell per session.

In [ ]:
import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal, spearmanr, pearsonr
from scipy.spatial.distance import pdist, squareform
import warnings
warnings.filterwarnings('ignore')

sns.set_context('talk')
sns.set_style('whitegrid')

# ── Constants ──
ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}

# ── Load GLM scores and cell-type labels from all mice and sessions ──
glm_records = []
for mouse_id in MOUSE_IDS:
    z = zarr.open(f'{ZARR_DIR}/{mouse_id}_multimodal_data.zarr', 'r')
    subclass_names = z['transcriptomics/cell_type/subclass_name'][:]
    n_total_cells = len(subclass_names)
    
    for sess in SESSIONS:
        glm = z[f'ophys/drifting_gratings/{sess}/glm']
        score_total = glm['score/score_total'][:]
        score_visual = glm['score/score_visual'][:]
        score_state = glm['score/score_state'][:]
        score_test = glm['score/score_test'][:]
        n_glm = len(score_total)
        
        # GLM may be fit on a subset of cells
        # Use index mapping if available; otherwise assume first n_glm cells
        for ci in range(n_glm):
            sc_idx = ci if ci < n_total_cells else 0
            glm_records.append({
                'mouse_id': mouse_id, 'session': sess,
                'cell_idx': ci,
                'subclass': subclass_names[sc_idx] if sc_idx < n_total_cells else 'unknown',
                'score_total': score_total[ci],
                'score_visual': score_visual[ci],
                'score_state': score_state[ci],
                'score_test': score_test[ci],
                'visual_fraction': score_visual[ci] / max(score_total[ci], 1e-6),
                'state_fraction': score_state[ci] / max(score_total[ci], 1e-6),
            })

glm_df = pd.DataFrame(glm_records)
glm_df['subclass_short'] = glm_df['subclass'].map(SUBCLASS_SHORT)
present_subclasses = [s for s in SUBCLASS_ORDER if s in glm_df['subclass'].unique()]

print(f"Total GLM records: {len(glm_df)} ({glm_df['mouse_id'].nunique()} mice × "
      f"{glm_df['session'].nunique()} sessions)")
print(f"\nGLM score_total summary per session:")
print(glm_df.groupby(['mouse_id', 'session'])['score_total'].describe()[['count', 'mean', 'max']].round(4))

## G2: Condition-Specific GLM Temporal Kernels

Extract per-condition GLM coefficient vectors (30 temporal basis functions per stimulus condition). Analyze how kernel shape varies with contrast, temporal frequency, and cell type.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# G2  Condition-Specific GLM Temporal Kernels
# ══════════════════════════════════════════════════════════════════════

# Load kernels from one mouse (778174, session 1) for detailed analysis
z = zarr.open(f'{ZARR_DIR}/778174_multimodal_data.zarr', 'r')
glm = z['ophys/drifting_gratings/session_1/glm']
subclass_names = z['transcriptomics/cell_type/subclass_name'][:]
n_glm_cells = glm['score/score_total'].shape[0]

# Extract contrast-context kernels (block 0, TF=1, direction=0)
contrasts = [0.05, 0.1, 0.2, 0.4, 0.8]
contrast_kernels = {}
for c in contrasts:
    key = f'coef_block_0.0_TF_1.0_contrast_{c}_direction_0.0'
    if key in glm['coef']:
        contrast_kernels[c] = glm['coef'][key][:]  # (n_glm_cells, 30)

# Extract TF-context kernels (block 1, contrast=0.8, direction=0)
tfs = [1.0, 2.0, 4.0, 8.0, 15.0]
tf_kernels = {}
for tf in tfs:
    key = f'coef_block_1.0_TF_{tf}_contrast_0.8_direction_0.0'
    if key in glm['coef']:
        tf_kernels[tf] = glm['coef'][key][:]  # (n_glm_cells, 30)

# Kernel time axis (30 basis functions, assume evenly spaced over ~3 s trial)
kernel_time = np.linspace(0, 3.0, 30)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1-2. Mean contrast kernels by subclass (population average)
for si, sc in enumerate(['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut']):
    ax = axes[0, si]
    mask = subclass_names[:n_glm_cells] == sc
    if mask.sum() == 0:
        continue
    for c in contrasts:
        if c in contrast_kernels:
            mean_kernel = contrast_kernels[c][mask].mean(axis=0)
            ax.plot(kernel_time, mean_kernel, label=f'c={c}', lw=1.5)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('GLM coefficient')
    ax.set_title(f'G2: Contrast Kernels — {SUBCLASS_SHORT[sc]}', fontweight='bold')
    ax.legend(fontsize=7)
    ax.axhline(0, color='k', ls='--', alpha=0.3)

# 3. TF kernels for L4/5 IT
ax = axes[0, 2]
mask = subclass_names[:n_glm_cells] == '006 L4/5 IT CTX Glut'
if mask.sum() > 0:
    for tf in tfs:
        if tf in tf_kernels:
            mean_kernel = tf_kernels[tf][mask].mean(axis=0)
            ax.plot(kernel_time, mean_kernel, label=f'TF={tf}', lw=1.5)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('GLM coefficient')
    ax.set_title('G2: TF Kernels — L4/5 IT', fontweight='bold')
    ax.legend(fontsize=7)
    ax.axhline(0, color='k', ls='--', alpha=0.3)

# 4. Kernel peak time vs contrast by subclass
ax = axes[1, 0]
for sc in present_subclasses:
    mask = subclass_names[:n_glm_cells] == sc
    if mask.sum() < 3:
        continue
    peak_times = []
    for c in contrasts:
        if c in contrast_kernels:
            mean_k = contrast_kernels[c][mask].mean(axis=0)
            pt = kernel_time[np.argmax(mean_k)]
            peak_times.append(pt)
        else:
            peak_times.append(np.nan)
    ax.plot(contrasts, peak_times, 'o-', color=SUBCLASS_COLORS[sc],
            label=SUBCLASS_SHORT[sc], lw=1.5, ms=4)
ax.set_xlabel('Contrast')
ax.set_ylabel('Peak kernel time (s)')
ax.set_title('G2: Kernel Peak Time vs Contrast', fontweight='bold')
ax.legend(fontsize=7)
ax.set_xscale('log')

# 5. Kernel amplitude (max coef) vs contrast
ax = axes[1, 1]
for sc in present_subclasses:
    mask = subclass_names[:n_glm_cells] == sc
    if mask.sum() < 3:
        continue
    amps = []
    for c in contrasts:
        if c in contrast_kernels:
            amps.append(contrast_kernels[c][mask].mean(axis=0).max())
        else:
            amps.append(np.nan)
    ax.plot(contrasts, amps, 'o-', color=SUBCLASS_COLORS[sc],
            label=SUBCLASS_SHORT[sc], lw=1.5, ms=4)
ax.set_xlabel('Contrast')
ax.set_ylabel('Peak kernel amplitude')
ax.set_title('G2: Kernel Amplitude vs Contrast', fontweight='bold')
ax.legend(fontsize=7)
ax.set_xscale('log')

# 6. PCA of all kernels colored by subclass
ax = axes[1, 2]
from sklearn.decomposition import PCA
all_kernels = []
all_labels = []
for c in contrasts:
    if c in contrast_kernels:
        all_kernels.append(contrast_kernels[c])
        all_labels.extend([f'c={c}'] * n_glm_cells)
if len(all_kernels) > 0:
    K = np.vstack(all_kernels)
    pca = PCA(n_components=2)
    K_pca = pca.fit_transform(K)
    sc_repeated = np.tile(subclass_names[:n_glm_cells], len(all_kernels) // n_glm_cells + 1)[:len(K)]
    for sc in present_subclasses:
        mask = sc_repeated == sc
        if mask.sum() > 0:
            ax.scatter(K_pca[mask, 0], K_pca[mask, 1], c=SUBCLASS_COLORS[sc],
                       alpha=0.2, s=5, label=SUBCLASS_SHORT[sc])
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    ax.set_title('G2: Kernel Shape PCA', fontweight='bold')
    ax.legend(fontsize=6, markerscale=3)

plt.tight_layout()
plt.show()